# Assignment 10: MT19937 Stream Cipher and Predictable Token Attack

## Part A: MT19937-Based Stream Cipher and Seed Recovery

### Objective
- Understand how MT19937 can be used to generate a pseudorandom keystream.
- Implement a toy stream cipher using XOR encryption.
- Verify encryption and decryption using the same seed.
- Exploit known plaintext consisting of 14 consecutive `A` characters.
- Brute-force all 65,536 possible 16-bit seeds to recover the original seed.
- Verify the recovered seed by decrypting the complete ciphertext.

## 1. Understanding MT19937

MT19937 (Mersenne Twister) is a pseudorandom number generator (PRNG).

Key characteristics:
- It has a state size of 624 words, with each word containing 32 bits.
- It uses a recurrence relation to update its internal state.
- It applies a tempering operation to produce output values.
- It has a period of \(2^{19937}-1\).
- It produces statistically good pseudorandom sequences.

### Why is MT19937 not cryptographically secure?

Despite its good statistical properties, MT19937 is not designed for cryptographic security.

- Its internal state can be reconstructed from sufficient observed outputs.
- Its output is deterministic when the seed and state are known.
- It is unsuitable for generating cryptographic keys, secure tokens, or encryption keystreams.

In this assignment, we demonstrate how a small seed space makes brute-force recovery practical.

## 2. MT19937-Based Stream Cipher

A stream cipher encrypts data by combining plaintext with a generated keystream.

In this assignment:
1. MT19937 generates pseudorandom bytes using a seed.
2. Each keystream byte is XORed with the corresponding plaintext byte.
3. The resulting bytes form the ciphertext.

### Encryption

\[
C_i = P_i \oplus K_i
\]

Where:
- \(P_i\) = plaintext byte
- \(K_i\) = keystream byte
- \(C_i\) = ciphertext byte

### Decryption

\[
P_i = C_i \oplus K_i
\]

XOR is reversible because:

\[
(A \oplus B) \oplus B = A
\]

Therefore, the same function can perform both encryption and decryption, provided the same seed and keystream generation procedure are used.

## 3. Known Plaintext Attack

A known plaintext attack occurs when an attacker knows some part of the original plaintext and has access to its corresponding ciphertext.

For this assignment:
- The attacker knows that the plaintext contains 14 consecutive `A` characters.
- The location of this sequence is unknown.
- The encryption seed is a randomly selected 16-bit integer.

### How can the known plaintext help?

Since:

\[
C_i = P_i \oplus K_i
\]

We can calculate the corresponding keystream bytes:

\[
K_i = C_i \oplus P_i
\]

For the known sequence:

\[
K_i = C_i \oplus \texttt{0x41}
\]

The attacker can use this information to identify the correct seed by testing possible seeds and checking whether the decrypted plaintext contains 14 consecutive `A` characters.

### Search space

A 16-bit seed has:

\[
2^{16} = 65,536
\]

possible values.

Therefore, the attacker can test every possible seed instead of needing to guess the original one.

In [1]:
import numpy as np
import secrets
import string


def generate_keystream(seed: int, length: int) -> bytes:
    bit_generator = np.random.MT19937(seed)
    rng = np.random.Generator(bit_generator)

    return rng.integers(
        0, 256, size=length, dtype=np.uint8
    ).tobytes()


def xor_cipher(data: bytes, seed: int) -> bytes:
    keystream = generate_keystream(seed, len(data))
    return bytes(a ^ b for a, b in zip(data, keystream))

In [2]:
# Generate a random prefix of 5–20 printable characters
prefix_length = secrets.randbelow(16) + 5

printable_chars = (
    string.ascii_letters + string.digits + string.punctuation + " "
)

prefix = ''.join(
    secrets.choice(printable_chars)
    for _ in range(prefix_length)
)

# Append 14 consecutive A characters
plaintext = (prefix + "A" * 14).encode("utf-8")

# Generate a random 16-bit seed
original_seed = secrets.randbelow(65536)

# Encrypt the plaintext
ciphertext = xor_cipher(plaintext, original_seed)

print("Original plaintext:", plaintext.decode())
print("Original seed:", original_seed)
print("Plaintext length:", len(plaintext), "bytes")
print("Ciphertext (hex):", ciphertext.hex())

Original plaintext: vr+NEy=F,-N*(PNyAAAAAAAAAAAAAA
Original seed: 46611
Plaintext length: 30 bytes
Ciphertext (hex): 6ee8d5380088995aa41bc469ff1a54c650ef1b00c1731d84c53a66f8005e


In [3]:
def recover_seed(ciphertext: bytes):
    known_sequence = b"A" * 14
    matching_candidates = []

    for candidate_seed in range(65536):
        decrypted = xor_cipher(ciphertext, candidate_seed)

        if known_sequence in decrypted:
            matching_candidates.append(
                (candidate_seed, decrypted)
            )

    return matching_candidates


matches = recover_seed(ciphertext)

print("Total seeds tested:", 65536)
print("Matching candidates:", len(matches))

for seed, recovered_plaintext in matches:
    print("Recovered seed:", seed)
    print("Recovered plaintext:", recovered_plaintext.decode())

Total seeds tested: 65536
Matching candidates: 1
Recovered seed: 46611
Recovered plaintext: vr+NEy=F,-N*(PNyAAAAAAAAAAAAAA


In [4]:
recovered_seed, recovered_plaintext = matches[0]

# Decrypt the complete ciphertext using the recovered seed
verified_plaintext = xor_cipher(ciphertext, recovered_seed)

print("Original seed:", original_seed)
print("Recovered seed:", recovered_seed)
print("Seed recovery successful:", recovered_seed == original_seed)

print("\nOriginal plaintext:", plaintext.decode())
print("Recovered plaintext:", verified_plaintext.decode())

print("\nPlaintexts match:", verified_plaintext == plaintext)
print("14 consecutive A's found:", b"A" * 14 in verified_plaintext)
print("Complete ciphertext decrypted:", len(verified_plaintext) == len(ciphertext))

Original seed: 46611
Recovered seed: 46611
Seed recovery successful: True

Original plaintext: vr+NEy=F,-N*(PNyAAAAAAAAAAAAAA
Recovered plaintext: vr+NEy=F,-N*(PNyAAAAAAAAAAAAAA

Plaintexts match: True
14 consecutive A's found: True
Complete ciphertext decrypted: True


## 4. Experimental Results and Observations

The experiment successfully demonstrated seed recovery against an MT19937-based stream cipher.

| Parameter | Result |
|---|---|
| Original seed | 46611 |
| Seed size | 16 bits |
| Total candidate seeds tested | 65,536 |
| Matching candidates | 1 |
| Recovered seed | 46611 |
| Known plaintext sequence | 14 consecutive A characters |
| Plaintext recovery | Successful |

### Observations

1. Encryption and decryption were successful using the same seed.
2. The known plaintext sequence allowed us to identify the correct seed.
3. Brute-forcing all 65,536 possible seeds recovered the original seed.
4. The recovered seed decrypted the entire ciphertext correctly.
5. The experiment demonstrates why a small seed space and a predictable PRNG are unsuitable for cryptographic applications.

### Conclusion

MT19937 is suitable for general-purpose pseudorandom number generation but is not cryptographically secure. A small seed space makes exhaustive seed searching practical, allowing an attacker with known plaintext to recover the seed and decrypt the complete message.